# Semana 10: Espacios Vectoriales y Transformaciones Lineales
## Notebook Conceptual (NB1) - Transformando el Espacio con Datos Dummy

### Propósito de la sesión
Entender que las capas de una red neuronal son **transformaciones lineales** que deforman el espacio para hacer los datos separables. Visualizaremos cómo una matriz transforma una nube de puntos, demostraremos que una transformación lineal **no puede** separar círculos concéntricos, y veremos cómo la introducción de una **no-linealidad (ReLU)** permite lograr la separación.

### Objetivos de aprendizaje
*   Visualizar transformaciones lineales en 2D: rotación, escalado, cizallamiento.
*   Comprender que una transformación lineal es una combinación de estas operaciones básicas.
*   Demostrar que los círculos concéntricos (no linealmente separables) siguen siendo no separables tras cualquier transformación lineal.
*   Introducir una **no-linealidad (ReLU)** y observar cómo transforma el espacio para permitir la separación.
*   Conectar estos conceptos con capas de redes neuronales, embeddings y capas convolucionales.

## Configuración Inicial

Importamos las librerías necesarias: NumPy para operaciones, Matplotlib para visualizaciones.

In [ ]:
# Importamos librerías
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle

# Configuración de matplotlib
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 6)

# Fijamos semilla para reproducibilidad
np.random.seed(42)

print("🎯 Librerías importadas correctamente.")

---
## 1. Datos: Círculos Concéntricos (No Linealmente Separables)

Generamos dos círculos concéntricos: uno interior (clase 0) y otro exterior (clase 1). Este es un problema clásico que **no** puede resolverse con un modelo lineal.

In [ ]:
def generate_concentric_circles(n_samples=500, r1=1.0, r2=2.0, noise=0.05):
    """
    Genera dos círculos concéntricos.
    Clase 0: círculo interior de radio r1
    Clase 1: círculo exterior de radio r2
    """
    # Clase 0 (interior)
    theta0 = np.random.uniform(0, 2*np.pi, n_samples//2)
    r0 = r1 + np.random.randn(n_samples//2) * noise
    x0 = r0 * np.cos(theta0)
    y0 = r0 * np.sin(theta0)
    
    # Clase 1 (exterior)
    theta1 = np.random.uniform(0, 2*np.pi, n_samples//2)
    r1_vals = r2 + np.random.randn(n_samples//2) * noise
    x1 = r1_vals * np.cos(theta1)
    y1 = r1_vals * np.sin(theta1)
    
    X = np.vstack([np.column_stack([x0, y0]), np.column_stack([x1, y1])])
    y = np.hstack([np.zeros(n_samples//2), np.ones(n_samples//2)])
    
    return X, y

X, y = generate_concentric_circles(n_samples=500, r1=1.0, r2=2.0, noise=0.1)

plt.figure(figsize=(8, 8))
plt.scatter(X[y==0, 0], X[y==0, 1], c='blue', label='Clase 0 (interior)', alpha=0.6, edgecolors='k')
plt.scatter(X[y==1, 0], X[y==1, 1], c='red', label='Clase 1 (exterior)', alpha=0.6, edgecolors='k')
plt.xlabel('x₁')
plt.ylabel('x₂')
plt.title('Círculos Concéntricos: No Linealmente Separables')
plt.legend()
plt.grid(True)
plt.axis('equal')
plt.show()

print("📌 No se puede dibujar una línea recta que separe las dos clases.")

---
## 2. Transformaciones Lineales en 2D

Una transformación lineal en 2D está definida por una matriz $\mathbf{W} \in \mathbb{R}^{2\times 2}$:

$$\mathbf{x}' = \mathbf{W} \mathbf{x}$$

Visualicemos diferentes transformaciones lineales aplicadas a los círculos concéntricos.

In [ ]:
def apply_transform(X, W):
    """Aplica transformación lineal W a los puntos X."""
    return X @ W.T

def plot_transform(X, y, W, title):
    X_trans = apply_transform(X, W)
    
    plt.figure(figsize=(14, 6))
    
    plt.subplot(1, 2, 1)
    plt.scatter(X[y==0, 0], X[y==0, 1], c='blue', alpha=0.6, edgecolors='k')
    plt.scatter(X[y==1, 0], X[y==1, 1], c='red', alpha=0.6, edgecolors='k')
    plt.xlabel('x₁')
    plt.ylabel('x₂')
    plt.title('Original')
    plt.grid(True)
    plt.axis('equal')
    
    plt.subplot(1, 2, 2)
    plt.scatter(X_trans[y==0, 0], X_trans[y==0, 1], c='blue', alpha=0.6, edgecolors='k')
    plt.scatter(X_trans[y==1, 0], X_trans[y==1, 1], c='red', alpha=0.6, edgecolors='k')
    plt.xlabel('x₁\'')
    plt.ylabel('x₂\'')
    plt.title(f'Transformada: {title}')
    plt.grid(True)
    plt.axis('equal')
    
    plt.tight_layout()
    plt.show()

# Rotación de 45 grados
theta = np.radians(45)
W_rot = np.array([[np.cos(theta), -np.sin(theta)],
                  [np.sin(theta), np.cos(theta)]])
plot_transform(X, y, W_rot, 'Rotación 45°')

# Escalado (estirar en x, comprimir en y)
W_scale = np.array([[2.0, 0.0],
                    [0.0, 0.5]])
plot_transform(X, y, W_scale, 'Escalado (2x, 0.5y)')

# Cizallamiento horizontal (shear)
W_shear = np.array([[1.0, 1.5],
                    [0.0, 1.0]])
plot_transform(X, y, W_shear, 'Cizallamiento horizontal')

# Transformación arbitraria (combinación)
W_arb = np.array([[1.2, 0.8],
                  [-0.5, 1.1]])
plot_transform(X, y, W_arb, 'Transformación arbitraria')

### 2.1. Análisis

Observamos que, aunque los puntos se deforman (rotan, estiran, cizallan), **las dos clases siguen siendo anillos concéntricos transformados** (elipses concéntricas). **Ninguna transformación lineal puede separarlas** porque la transformación lineal preserva la estructura de "convexidad" y no puede desenredar los anillos.

---
## 3. Demostración: Una Transformación Lineal No Puede Separar

Intentemos buscar una transformación lineal que separe las clases. Podemos pensar en un clasificador lineal como una línea (o hiperplano) que separa. Una transformación lineal seguida de un clasificador lineal es **equivalente** a un solo clasificador lineal. Por lo tanto, si los datos no son linealmente separables originalmente, tampoco lo serán después de cualquier transformación lineal.

In [ ]:
# Función para visualizar un clasificador lineal (frontera de decisión)
def plot_linear_classifier(X, y, w, b, title):
    plt.figure(figsize=(8, 8))
    plt.scatter(X[y==0, 0], X[y==0, 1], c='blue', alpha=0.6, edgecolors='k', label='Clase 0')
    plt.scatter(X[y==1, 0], X[y==1, 1], c='red', alpha=0.6, edgecolors='k', label='Clase 1')
    
    # Dibujar la línea w·x + b = 0
    x_min, x_max = plt.xlim()
    y_min, y_max = plt.ylim()
    
    # Resolver para dos puntos
    if w[1] != 0:
        x_vals = np.array([x_min, x_max])
        y_vals = -(w[0] * x_vals + b) / w[1]
        plt.plot(x_vals, y_vals, 'k--', linewidth=2, label='Frontera lineal')
    else:
        # w[1] == 0, línea vertical
        x_line = -b / w[0]
        plt.axvline(x=x_line, color='k', linestyle='--', linewidth=2, label='Frontera lineal')
    
    plt.xlabel('x₁')
    plt.ylabel('x₂')
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.axis('equal')
    plt.show()

# Intentamos un clasificador lineal en los datos originales (obviamente no funcionará)
w_lin = np.array([1.0, 0.0])  # frontera vertical
b_lin = 0.0
plot_linear_classifier(X, y, w_lin, b_lin, 'Clasificador lineal en datos originales')

# Probamos en los datos transformados (por ejemplo, con la transformación arbitraria)
X_trans_arb = apply_transform(X, W_arb)
plot_linear_classifier(X_trans_arb, y, w_lin, b_lin, 'Clasificador lineal en datos transformados')

print("📌 En ambos casos, una sola línea no puede separar las clases.")

---
## 4. Introduciendo No-linealidad: ReLU

En una red neuronal, después de la transformación lineal aplicamos una función de activación no lineal, como ReLU: $\text{ReLU}(x) = \max(0, x)$.

Esto permite que el espacio se "pliegue" y pueda separar datos que antes no eran separables linealmente.

In [ ]:
def relu(x):
    return np.maximum(0, x)

# Aplicamos transformación lineal + ReLU
def apply_transform_nonlinear(X, W):
    X_linear = X @ W.T
    return relu(X_linear)

# Probamos con la transformación arbitraria más ReLU
X_trans_relu = apply_transform_nonlinear(X, W_arb)

plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
plt.scatter(X[y==0, 0], X[y==0, 1], c='blue', alpha=0.6, edgecolors='k')
plt.scatter(X[y==1, 0], X[y==1, 1], c='red', alpha=0.6, edgecolors='k')
plt.xlabel('x₁')
plt.ylabel('x₂')
plt.title('Original')
plt.grid(True)
plt.axis('equal')

plt.subplot(1, 2, 2)
plt.scatter(X_trans_relu[y==0, 0], X_trans_relu[y==0, 1], c='blue', alpha=0.6, edgecolors='k')
plt.scatter(X_trans_relu[y==1, 0], X_trans_relu[y==1, 1], c='red', alpha=0.6, edgecolors='k')
plt.xlabel('x₁\' después de ReLU')
plt.ylabel('x₂\' después de ReLU')
plt.title('Transformación lineal + ReLU')
plt.grid(True)
plt.axis('equal')

plt.tight_layout()
plt.show()

print("📌 ReLU introduce no-linealidad: aplasta los valores negativos a cero, creando pliegues.")

### 4.1. ¿Podemos ahora separar con una línea?

Veamos si después de aplicar transformación lineal + ReLU, las clases se vuelven linealmente separables.

In [ ]:
# Intentamos un clasificador lineal en el espacio transformado con ReLU
plot_linear_classifier(X_trans_relu, y, w_lin, b_lin, 'Clasificador lineal después de ReLU')

# A veces una simple línea vertical/horizontal no es suficiente; probemos una orientación diferente
w_lin2 = np.array([-1.0, 1.0])
b_lin2 = 0.5
plot_linear_classifier(X_trans_relu, y, w_lin2, b_lin2, 'Clasificador lineal (otra orientación) después de ReLU')

print("📌 Dependiendo de la transformación y la no-linealidad, ahora podría ser posible separar.")
print("📌 En una red real, se aprenden tanto W como el clasificador final.")

---
## 5. Conexiones IA

### 5.1. Deep Learning: Capas Fully Connected
Cada capa de una red neuronal realiza: $\mathbf{h} = \sigma(\mathbf{W}\mathbf{x} + \mathbf{b})$. Primero una transformación lineal ($\mathbf{W}\mathbf{x}$), luego una no-linealidad ($\sigma$). La composición de muchas capas permite aprender transformaciones complejas que separan los datos.

### 5.2. Visión por Computador: Capas Convolucionales
Las convoluciones también son transformaciones lineales (aunque con una estructura especial de matriz dispersa). Van seguidas de no-linealidades (ReLU).

### 5.3. NLP: Embeddings
Los embeddings son una transformación lineal de vectores one-hot a un espacio denso: $\mathbf{e}_{palabra} = \mathbf{E} \cdot \mathbf{one\_hot}$.

In [ ]:
# Ejemplo: embedding de palabras como transformación lineal
vocab_size = 5
embedding_dim = 3

# Matriz de embedding (cada columna es el embedding de una palabra)
E = np.random.randn(embedding_dim, vocab_size)

# Representación one-hot de la palabra "gato" (índice 2)
one_hot_gato = np.zeros(vocab_size)
one_hot_gato[2] = 1.0

embedding_gato = E @ one_hot_gato
print(f"Embedding de 'gato': {embedding_gato}")

---
## Ejercicios para el Estudiante

1.  **Experimenta con otras transformaciones:** Prueba matrices de transformación que sean singulares (determinante cero). ¿Qué ocurre con los puntos? ¿Por qué?

2.  **Otra no-linealidad:** Reemplaza ReLU por **tanh**. ¿Cómo cambia la forma de los puntos transformados? ¿Sigue siendo posible la separación?

3.  **Red de una capa oculta:** Implementa una red con una capa oculta de 2 neuronas y ReLU, seguida de un clasificador lineal. Entrena (manualmente con descenso de gradiente) para separar los círculos. (Puedes usar PyTorch).

4.  **Visualización de la deformación:** Para una transformación lineal $\mathbf{W}$ y una no-linealidad, visualiza cómo se deforma una malla de puntos regular. ¿Qué patrones observas?

5.  **Reflexión:** ¿Por qué las redes profundas necesitan no-linealidades? ¿Qué pasaría si todas las capas fueran lineales?

---
## Conclusiones de la Sesión

*   Las **transformaciones lineales** (rotación, escalado, cizallamiento) deforman el espacio pero **preservan la estructura lineal**: círculos concéntricos se convierten en elipses concéntricas, pero siguen siendo anidados y no separables por una línea.
*   Hemos demostrado visualmente que **ninguna transformación lineal puede separar clases que no son linealmente separables** (como los círculos concéntricos).
*   La introducción de una **no-linealidad** (ReLU) después de la transformación lineal permite "plegar" el espacio, creando regiones donde las clases pueden volverse separables.
*   Este principio es la base de las **redes neuronales**: cada capa es $\mathbf{h} = \sigma(\mathbf{W}\mathbf{x} + \mathbf{b})$, combinando una transformación lineal con una no-linealidad.
*   Conectamos estos conceptos con aplicaciones en **Deep Learning** (capas fully connected), **Visión por Computador** (capas convolucionales) y **NLP** (embeddings como transformaciones lineales).

¡Ahora entiendes por qué las redes neuronales necesitan activaciones no lineales para resolver problemas complejos!